# 04 - Modelagem da Camada Gold

## Objetivo

Construir a camada Gold do pipeline por meio de uma modelagem dimensional em esquema estrela.

A granularidade definida para a tabela fato é de uma linha por ocorrência de acidente.

A modelagem foi estruturada para separar as medidas quantitativas dos acidentes de seus respectivos contextos temporais, geográficos, rodoviários e circunstanciais.

Serão criadas as seguintes estruturas:

- `fato_acidentes`;
- `dim_data`;
- `dim_localizacao`;
- `dim_via`;
- `dim_acidente`;
- `dim_condicoes`;
- `dim_tracado`;
- `ponte_acidente_tracado`.

A coluna `tracado_via` exige tratamento específico por permitir múltiplas características em uma mesma ocorrência. Por esse motivo, será utilizada uma tabela ponte entre acidentes e características de traçado.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver = spark.table(
    "workspace.default.prf_acidentes_silver"
)

print(f"Registros Silver: {df_silver.count()}")
print(f"Colunas Silver: {len(df_silver.columns)}")

Registros Silver: 342624
Colunas Silver: 38


## Criação da dimensão de data

A dimensão `dim_data` será responsável por armazenar os atributos temporais das ocorrências.

Ela permitirá realizar análises por:

- data;
- ano;
- mês;
- dia da semana;
- hora;
- fim de semana;
- fase do dia.

Cada combinação distinta desses atributos receberá uma chave substituta (`id_data`), que posteriormente será utilizada na tabela fato.

In [0]:
dim_data_base = (
    df_silver.select(
        "data_inversa",
        "ano",
        "mes",
        "dia_semana",
        "hora",
        "fim_de_semana",
        "fase_dia"
    )
    .dropDuplicates()
)

In [0]:
print(f"Registros distintos na dimensão de data: {dim_data_base.count()}")

Registros distintos na dimensão de data: 59739


In [0]:
window_data = Window.orderBy(
    "data_inversa",
    "hora",
    "fase_dia"
)

dim_data = (
    dim_data_base
    .withColumn(
        "id_data",
        F.row_number().over(window_data)
    )
    .select(
        "id_data",
        "data_inversa",
        "ano",
        "mes",
        "dia_semana",
        "hora",
        "fim_de_semana",
        "fase_dia"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(dim_data.limit(20))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


id_data,data_inversa,ano,mes,dia_semana,hora,fim_de_semana,fase_dia
1,2021-01-01,2021,1,sexta-feira,0,0,Plena Noite
2,2021-01-01,2021,1,sexta-feira,1,0,Plena Noite
3,2021-01-01,2021,1,sexta-feira,2,0,Plena Noite
4,2021-01-01,2021,1,sexta-feira,3,0,Amanhecer
5,2021-01-01,2021,1,sexta-feira,3,0,Plena Noite
6,2021-01-01,2021,1,sexta-feira,4,0,Amanhecer
7,2021-01-01,2021,1,sexta-feira,4,0,Plena Noite
8,2021-01-01,2021,1,sexta-feira,5,0,Amanhecer
9,2021-01-01,2021,1,sexta-feira,5,0,Plena Noite
10,2021-01-01,2021,1,sexta-feira,5,0,Pleno dia


In [0]:
print(f"Total de linhas: {dim_data.count()}")
print(
    "IDs distintos:",
    dim_data.select("id_data").distinct().count()
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Total de linhas: 59739
IDs distintos: 59739


In [0]:
duplicidades_dim_data = (
    dim_data
    .groupBy(
        "data_inversa",
        "ano",
        "mes",
        "dia_semana",
        "hora",
        "fim_de_semana",
        "fase_dia"
    )
    .count()
    .filter(F.col("count") > 1)
)

print(
    f"Duplicidades na dimensão de data: "
    f"{duplicidades_dim_data.count()}"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Duplicidades na dimensão de data: 0


### Resultado da criação da `dim_data`

A dimensão `dim_data` foi criada a partir das combinações temporais distintas presentes na camada Silver.

Foi adicionada uma chave substituta denominada `id_data`, permitindo que a tabela fato faça referência aos atributos temporais sem necessidade de repetir essas informações diretamente.

Foram realizadas validações de unicidade da chave e das combinações temporais, garantindo que cada registro da dimensão represente uma combinação única de data, hora e demais características temporais.

In [0]:
(
    dim_data.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.dim_data")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
spark.table("workspace.default.dim_data").count()

59739

### Observação sobre a criação da chave substituta

Durante a geração da chave `id_data` com `row_number()`, o Spark apresentou um aviso de performance informando que a operação de janela foi executada sem particionamento.

Como a dimensão de data possui volume significativamente menor que a tabela de ocorrências, o aviso não comprometeu a execução do MVP.

Em cenários de maior escala, outras estratégias de geração de chaves substitutas poderiam ser consideradas para evitar concentração do processamento em uma única partição.

## Criação da dimensão de localização

A dimensão `dim_localizacao` armazenará os atributos geográficos e administrativos relacionados ao local da ocorrência.

Serão utilizados os seguintes campos:

- UF;
- município;
- latitude;
- longitude;
- regional;
- delegacia;
- UOP.

Cada combinação distinta receberá uma chave substituta `id_localizacao`, posteriormente utilizada na tabela fato.

In [0]:
dim_localizacao_base = (
    df_silver.select(
        "uf",
        "municipio",
        "latitude",
        "longitude",
        "regional",
        "delegacia",
        "uop"
    )
    .dropDuplicates()
)

In [0]:
print(
    f"Registros distintos na dimensão de localização: "
    f"{dim_localizacao_base.count()}"
)

Registros distintos na dimensão de localização: 173380


In [0]:
window_localizacao = Window.orderBy(
    "uf",
    "municipio",
    "latitude",
    "longitude",
    "regional",
    "delegacia",
    "uop"
)

dim_localizacao = (
    dim_localizacao_base
    .withColumn(
        "id_localizacao",
        F.row_number().over(window_localizacao)
    )
    .select(
        "id_localizacao",
        "uf",
        "municipio",
        "latitude",
        "longitude",
        "regional",
        "delegacia",
        "uop"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
print(f"Total de linhas: {dim_localizacao.count()}")

print(
    "IDs distintos:",
    dim_localizacao.select("id_localizacao").distinct().count()
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Total de linhas: 173380
IDs distintos: 173380


In [0]:
duplicidades_dim_localizacao = (
    dim_localizacao
    .groupBy(
        "uf",
        "municipio",
        "latitude",
        "longitude",
        "regional",
        "delegacia",
        "uop"
    )
    .count()
    .filter(F.col("count") > 1)
)

print(
    f"Duplicidades na dimensão de localização: "
    f"{duplicidades_dim_localizacao.count()}"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Duplicidades na dimensão de localização: 0


In [0]:
display(dim_localizacao.limit(20))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


id_localizacao,uf,municipio,latitude,longitude,regional,delegacia,uop
1,AC,ACRELANDIA,-9.95837031,-67.0930509,SPRF-AC,DEL01-AC,UOP01-DEL01-AC
2,AC,ACRELANDIA,-9.95520038,-67.08448385,SPRF-AC,DEL01-AC,UOP03-DEL01-AC
3,AC,ACRELANDIA,-9.95203666,-67.07591312,SPRF-AC,DEL01-AC,UOP03-DEL01-AC
4,AC,ACRELANDIA,-9.94887291,-67.06734203,SPRF-AC,DEL01-AC,UOP01-DEL01-AC
5,AC,ACRELANDIA,-9.94570874,-67.0587711,SPRF-AC,DEL01-AC,UOP03-DEL01-AC
6,AC,ACRELANDIA,-9.93938587,-67.04162805,SPRF-AC,DEL01-AC,UOP01-DEL01-AC
7,AC,ACRELANDIA,-9.938445,-67.027195,SPRF-AC,DEL01-AC,UOP01-DEL01-AC
8,AC,ACRELANDIA,-9.938263,-67.02349,SPRF-AC,DEL01-AC,UOP03-DEL01-AC
9,AC,ACRELANDIA,-9.93817553,-67.01453399,SPRF-AC,DEL01-AC,UOP01-DEL01-AC
10,AC,ACRELANDIA,-9.93810945,-67.00725699,SPRF-AC,DEL01-AC,UOP01-DEL01-AC


In [0]:
(
    dim_localizacao.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.dim_localizacao")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
spark.table("workspace.default.dim_localizacao").count()

173380

## Criação da dimensão de via

A dimensão `dim_via` armazenará os atributos relacionados à rodovia e às características básicas da via onde ocorreu o acidente.

Serão utilizados os seguintes campos:

- BR;
- km;
- sentido da via;
- tipo de pista;
- uso do solo.

Cada combinação distinta receberá uma chave substituta `id_via`, posteriormente utilizada na tabela fato.

A coluna `tracado_via` não será incluída nesta dimensão porque pode conter múltiplas características em uma mesma ocorrência. Esse atributo será tratado separadamente por meio de uma dimensão específica e uma tabela ponte.

In [0]:
dim_via_base = (
    df_silver.select(
        "br",
        "km",
        "sentido_via",
        "tipo_pista",
        "uso_solo"
    )
    .dropDuplicates()
)

In [0]:
print(
    f"Registros distintos na dimensão de via: "
    f"{dim_via_base.count()}"
)

Registros distintos na dimensão de via: 165129


In [0]:
window_via = Window.orderBy(
    "br",
    "km",
    "sentido_via",
    "tipo_pista",
    "uso_solo"
)

dim_via = (
    dim_via_base
    .withColumn(
        "id_via",
        F.row_number().over(window_via)
    )
    .select(
        "id_via",
        "br",
        "km",
        "sentido_via",
        "tipo_pista",
        "uso_solo"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
print(f"Total de linhas: {dim_via.count()}")

print(
    "IDs distintos:",
    dim_via.select("id_via").distinct().count()
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Total de linhas: 165129
IDs distintos: 165129


In [0]:
duplicidades_dim_via = (
    dim_via
    .groupBy(
        "br",
        "km",
        "sentido_via",
        "tipo_pista",
        "uso_solo"
    )
    .count()
    .filter(F.col("count") > 1)
)

print(
    f"Duplicidades na dimensão de via: "
    f"{duplicidades_dim_via.count()}"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Duplicidades na dimensão de via: 0


In [0]:
display(dim_via.limit(20))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


id_via,br,km,sentido_via,tipo_pista,uso_solo
1,0,0.0,null,Dupla,Não
2,0,0.0,null,Dupla,Sim
3,0,0.0,null,Múltipla,Não
4,0,0.0,null,Múltipla,Sim
5,0,0.0,null,Simples,Não
6,0,0.0,null,Simples,Sim
7,10,0.0,Crescente,Simples,Não
8,10,0.0,Decrescente,Simples,Não
9,10,0.0,Decrescente,Simples,Sim
10,10,0.2,Decrescente,Simples,Não


In [0]:
(
    dim_via.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.dim_via")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
spark.table("workspace.default.dim_via").count()

165129

## Criação da dimensão de acidente

A dimensão `dim_acidente` armazenará os atributos que descrevem a natureza e a classificação da ocorrência.

Serão utilizados os seguintes campos:

- causa do acidente;
- tipo do acidente;
- classificação do acidente.

Cada combinação distinta receberá uma chave substituta `id_acidente_dim`, posteriormente utilizada na tabela fato.

Essa separação permite reduzir a repetição de atributos categóricos na tabela fato e facilita análises por causa, tipo e classificação do acidente.

In [0]:
dim_acidente_base = (
    df_silver.select(
        "causa_acidente",
        "tipo_acidente",
        "classificacao_acidente"
    )
    .dropDuplicates()
)

In [0]:
print(
    f"Registros distintos na dimensão de acidente: "
    f"{dim_acidente_base.count()}"
)

Registros distintos na dimensão de acidente: 2387


In [0]:
window_acidente = Window.orderBy(
    "causa_acidente",
    "tipo_acidente",
    "classificacao_acidente"
)

dim_acidente = (
    dim_acidente_base
    .withColumn(
        "id_acidente_dim",
        F.row_number().over(window_acidente)
    )
    .select(
        "id_acidente_dim",
        "causa_acidente",
        "tipo_acidente",
        "classificacao_acidente"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
print(f"Total de linhas: {dim_acidente.count()}")

print(
    "IDs distintos:",
    dim_acidente.select("id_acidente_dim").distinct().count()
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Total de linhas: 2387
IDs distintos: 2387


In [0]:
duplicidades_dim_acidente = (
    dim_acidente
    .groupBy(
        "causa_acidente",
        "tipo_acidente",
        "classificacao_acidente"
    )
    .count()
    .filter(F.col("count") > 1)
)

print(
    f"Duplicidades na dimensão de acidente: "
    f"{duplicidades_dim_acidente.count()}"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Duplicidades na dimensão de acidente: 0


In [0]:
display(dim_acidente.limit(20))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


id_acidente_dim,causa_acidente,tipo_acidente,classificacao_acidente
1,Acessar a via sem observar a presença dos outros veículos,Atropelamento de Animal,Com Vítimas Fatais
2,Acessar a via sem observar a presença dos outros veículos,Atropelamento de Animal,Com Vítimas Feridas
3,Acessar a via sem observar a presença dos outros veículos,Atropelamento de Animal,Sem Vítimas
4,Acessar a via sem observar a presença dos outros veículos,Atropelamento de Pedestre,Com Vítimas Fatais
5,Acessar a via sem observar a presença dos outros veículos,Atropelamento de Pedestre,Com Vítimas Feridas
6,Acessar a via sem observar a presença dos outros veículos,Atropelamento de Pedestre,Sem Vítimas
7,Acessar a via sem observar a presença dos outros veículos,Capotamento,Com Vítimas Feridas
8,Acessar a via sem observar a presença dos outros veículos,Capotamento,Sem Vítimas
9,Acessar a via sem observar a presença dos outros veículos,Colisão com objeto,Com Vítimas Fatais
10,Acessar a via sem observar a presença dos outros veículos,Colisão com objeto,Com Vítimas Feridas


In [0]:
(
    dim_acidente.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.dim_acidente")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
spark.table("workspace.default.dim_acidente").count()

2387

## Criação da dimensão de condições

A dimensão `dim_condicoes` armazenará atributos relacionados às condições ambientais observadas no momento da ocorrência.

Nesta etapa será utilizada a coluna:

- condição meteorológica.

Cada valor distinto receberá uma chave substituta `id_condicoes`, posteriormente utilizada na tabela fato.

A separação desse atributo em uma dimensão própria permite analisar a relação entre condições meteorológicas e gravidade dos acidentes sem repetir o valor textual diretamente na tabela fato.

In [0]:
dim_condicoes_base = (
    df_silver.select(
        "condicao_metereologica"
    )
    .dropDuplicates()
)

In [0]:
print(
    f"Registros distintos na dimensão de condições: "
    f"{dim_condicoes_base.count()}"
)

Registros distintos na dimensão de condições: 10


In [0]:
window_condicoes = Window.orderBy(
    "condicao_metereologica"
)

dim_condicoes = (
    dim_condicoes_base
    .withColumn(
        "id_condicoes",
        F.row_number().over(window_condicoes)
    )
    .select(
        "id_condicoes",
        "condicao_metereologica"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
print(f"Total de linhas: {dim_condicoes.count()}")

print(
    "IDs distintos:",
    dim_condicoes.select("id_condicoes").distinct().count()
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Total de linhas: 10
IDs distintos: 10


In [0]:
duplicidades_dim_condicoes = (
    dim_condicoes
    .groupBy(
        "condicao_metereologica"
    )
    .count()
    .filter(F.col("count") > 1)
)

print(
    f"Duplicidades na dimensão de condições: "
    f"{duplicidades_dim_condicoes.count()}"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Duplicidades na dimensão de condições: 0


In [0]:
display(dim_condicoes)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


id_condicoes,condicao_metereologica
1,null
2,Chuva
3,Céu Claro
4,Garoa/Chuvisco
5,Granizo
6,Neve
7,Nevoeiro/Neblina
8,Nublado
9,Sol
10,Vento


In [0]:
(
    dim_condicoes.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.dim_condicoes")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
spark.table("workspace.default.dim_condicoes").count()

10

### Resultado da criação da `dim_condicoes`

A dimensão `dim_condicoes` foi criada a partir dos valores distintos de condição meteorológica presentes na camada Silver.

Cada condição recebeu uma chave substituta `id_condicoes`, permitindo que a tabela fato faça referência às condições ambientais sem armazenar repetidamente os valores textuais.

Os registros sem informação meteorológica permaneceram representados como valores nulos, preservando a distinção entre ausência de informação e categorias válidas.

## Criação da dimensão de traçado e tabela ponte

Durante a análise de qualidade foi identificado que a coluna `tracado_via` pode armazenar múltiplas características no mesmo registro, separadas por ponto e vírgula.

Como uma ocorrência pode estar associada a mais de uma característica de traçado, esse atributo não será armazenado diretamente em uma dimensão tradicional.

Será criada:

- uma dimensão `dim_tracado`, contendo uma linha por característica individual de traçado;
- uma tabela `ponte_acidente_tracado`, responsável por relacionar cada ocorrência às características de traçado associadas a ela.

Essa estrutura permite representar corretamente a relação muitos-para-muitos sem alterar a granularidade da tabela fato.

In [0]:
df_tracado_explodido = (
    df_silver
    .select(
        "id",
        F.explode("tracado_via_lista").alias("caracteristica_tracado")
    )
    .withColumn(
        "caracteristica_tracado",
        F.trim(F.col("caracteristica_tracado"))
    )
)

In [0]:
dim_tracado_base = (
    df_tracado_explodido
    .select("caracteristica_tracado")
    .dropDuplicates()
)

In [0]:
print(
    f"Características distintas de traçado: "
    f"{dim_tracado_base.count()}"
)

Características distintas de traçado: 12


In [0]:
window_tracado = Window.orderBy(
    "caracteristica_tracado"
)

dim_tracado = (
    dim_tracado_base
    .withColumn(
        "id_tracado",
        F.row_number().over(window_tracado)
    )
    .select(
        "id_tracado",
        "caracteristica_tracado"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
print(f"Total de linhas: {dim_tracado.count()}")

print(
    "IDs distintos:",
    dim_tracado.select("id_tracado").distinct().count()
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Total de linhas: 12
IDs distintos: 12


In [0]:
display(dim_tracado)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


id_tracado,caracteristica_tracado
1,Aclive
2,Curva
3,Declive
4,Desvio Temporário
5,Em Obras
6,Interseção de Vias
7,Ponte
8,Reta
9,Retorno Regulamentado
10,Rotatória


In [0]:
(
    dim_tracado.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.dim_tracado")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
spark.table("workspace.default.dim_tracado").count()

12

In [0]:
ponte_acidente_tracado = (
    df_tracado_explodido
    .join(
        dim_tracado,
        on="caracteristica_tracado",
        how="left"
    )
    .select(
        F.col("id").alias("id_acidente"),
        "id_tracado"
    )
    .dropDuplicates()
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
print(
    f"Relacionamentos acidente-traçado: "
    f"{ponte_acidente_tracado.count()}"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Relacionamentos acidente-traçado: 418561


In [0]:
print(
    "Relacionamentos sem id_tracado:",
    ponte_acidente_tracado
    .filter(F.col("id_tracado").isNull())
    .count()
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Relacionamentos sem id_tracado: 0


In [0]:
display(
    ponte_acidente_tracado
    .orderBy("id_acidente")
    .limit(30)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


id_acidente,id_tracado
331693,8
331694,8
331696,8
331699,2
331701,3
331702,12
331703,8
331704,8
331706,2
331707,3


In [0]:
(
    ponte_acidente_tracado.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.default.ponte_acidente_tracado"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
spark.table(
    "workspace.default.ponte_acidente_tracado"
).count()

418561

### Resultado da modelagem de `tracado_via`

A coluna multivalorada `tracado_via` foi normalizada em duas estruturas.

A dimensão `dim_tracado` contém as características individuais de traçado identificadas na base.

A tabela `ponte_acidente_tracado` relaciona cada ocorrência às características correspondentes por meio das chaves `id_acidente` e `id_tracado`.

Essa solução permite preservar corretamente acidentes associados a múltiplas características de via sem duplicar registros na tabela fato.

## Criação da tabela fato

A tabela `fato_acidentes` será a estrutura central da camada Gold.

Sua granularidade será de uma linha por ocorrência de acidente.

Ela armazenará:

- as medidas quantitativas do acidente;
- as chaves das dimensões temporal, geográfica, rodoviária, de acidente e de condições;
- campos de rastreabilidade da origem.

Os atributos descritivos serão obtidos por meio das dimensões, evitando repetição de informações textuais na tabela fato.

In [0]:
from pyspark.sql import functions as F

# Recarrega tudo diretamente das tabelas persistidas
fato_base = spark.table("workspace.default.prf_acidentes_silver")

d_data = spark.table("workspace.default.dim_data")
d_localizacao = spark.table("workspace.default.dim_localizacao")
d_via = spark.table("workspace.default.dim_via")
d_acidente = spark.table("workspace.default.dim_acidente")
d_condicoes = spark.table("workspace.default.dim_condicoes")


# 1. DATA
f = fato_base.alias("f")
d = d_data.alias("d")

fato_1 = (
    f.join(
        d,
        (F.col("f.data_inversa").eqNullSafe(F.col("d.data_inversa"))) &
        (F.col("f.ano").eqNullSafe(F.col("d.ano"))) &
        (F.col("f.mes").eqNullSafe(F.col("d.mes"))) &
        (F.col("f.dia_semana").eqNullSafe(F.col("d.dia_semana"))) &
        (F.col("f.hora").eqNullSafe(F.col("d.hora"))) &
        (F.col("f.fim_de_semana").eqNullSafe(F.col("d.fim_de_semana"))) &
        (F.col("f.fase_dia").eqNullSafe(F.col("d.fase_dia"))),
        "left"
    )
    .select("f.*", F.col("d.id_data"))
)


# 2. LOCALIZAÇÃO
f = fato_1.alias("f")
d = d_localizacao.alias("d")

fato_2 = (
    f.join(
        d,
        (F.col("f.uf").eqNullSafe(F.col("d.uf"))) &
        (F.col("f.municipio").eqNullSafe(F.col("d.municipio"))) &
        (F.col("f.latitude").eqNullSafe(F.col("d.latitude"))) &
        (F.col("f.longitude").eqNullSafe(F.col("d.longitude"))) &
        (F.col("f.regional").eqNullSafe(F.col("d.regional"))) &
        (F.col("f.delegacia").eqNullSafe(F.col("d.delegacia"))) &
        (F.col("f.uop").eqNullSafe(F.col("d.uop"))),
        "left"
    )
    .select("f.*", F.col("d.id_localizacao"))
)


# 3. VIA
f = fato_2.alias("f")
d = d_via.alias("d")

fato_3 = (
    f.join(
        d,
        (F.col("f.br").eqNullSafe(F.col("d.br"))) &
        (F.col("f.km").eqNullSafe(F.col("d.km"))) &
        (F.col("f.sentido_via").eqNullSafe(F.col("d.sentido_via"))) &
        (F.col("f.tipo_pista").eqNullSafe(F.col("d.tipo_pista"))) &
        (F.col("f.uso_solo").eqNullSafe(F.col("d.uso_solo"))),
        "left"
    )
    .select("f.*", F.col("d.id_via"))
)


# 4. ACIDENTE
f = fato_3.alias("f")
d = d_acidente.alias("d")

fato_4 = (
    f.join(
        d,
        (F.col("f.causa_acidente").eqNullSafe(F.col("d.causa_acidente"))) &
        (F.col("f.tipo_acidente").eqNullSafe(F.col("d.tipo_acidente"))) &
        (F.col("f.classificacao_acidente").eqNullSafe(F.col("d.classificacao_acidente"))),
        "left"
    )
    .select("f.*", F.col("d.id_acidente_dim"))
)


# 5. CONDIÇÕES
f = fato_4.alias("f")
d = d_condicoes.alias("d")

fato_5 = (
    f.join(
        d,
        F.col("f.condicao_metereologica").eqNullSafe(
            F.col("d.condicao_metereologica")
        ),
        "left"
    )
    .select("f.*", F.col("d.id_condicoes"))
)


# 6. Seleção final da tabela fato
fato_acidentes = (
    fato_5.select(
        F.col("id").alias("id_acidente"),
        "id_data",
        "id_localizacao",
        "id_via",
        "id_acidente_dim",
        "id_condicoes",
        "pessoas",
        "mortos",
        "feridos_leves",
        "feridos_graves",
        "feridos",
        "ilesos",
        "ignorados",
        "veiculos",
        "acidente_fatal",
        "ano_arquivo",
        "arquivo_origem"
    )
)

In [0]:
print("Registros Silver:", fato_base.count())
print("Registros Fato:", fato_acidentes.count())
print(
    "IDs distintos na Fato:",
    fato_acidentes.select("id_acidente").distinct().count()
)

Registros Silver: 342624
Registros Fato: 342624
IDs distintos na Fato: 342624


In [0]:
chaves_dimensoes = [
    "id_data",
    "id_localizacao",
    "id_via",
    "id_acidente_dim",
    "id_condicoes"
]

display(
    fato_acidentes.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in chaves_dimensoes
    ])
)

id_data,id_localizacao,id_via,id_acidente_dim,id_condicoes
0,0,0,0,0


In [0]:
chaves_dimensoes = [
    "id_data",
    "id_localizacao",
    "id_via",
    "id_acidente_dim",
    "id_condicoes"
]

display(
    fato_acidentes.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in chaves_dimensoes
    ])
)

id_data,id_localizacao,id_via,id_acidente_dim,id_condicoes
0,0,0,0,0


In [0]:
(
    fato_acidentes.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.fato_acidentes")
)

In [0]:
spark.table("workspace.default.fato_acidentes").count()

342624

### Resultado da criação da tabela fato

A tabela `fato_acidentes` foi criada com granularidade de uma linha por ocorrência.

A quantidade de registros foi validada em relação à camada Silver, mantendo os 342.624 acidentes originalmente presentes no conjunto tratado.

Também foi validada a unicidade do campo `id_acidente` e a integridade dos relacionamentos com as dimensões.

A tabela fato foi persistida em formato Delta no Unity Catalog do Databricks e passa a compor a camada Gold do pipeline.

In [0]:
from pyspark.sql import functions as F

fato = spark.table("workspace.default.fato_acidentes")

print("Registros na fato:", fato.count())
print(
    "IDs distintos:",
    fato.select("id_acidente").distinct().count()
)

chaves = [
    "id_data",
    "id_localizacao",
    "id_via",
    "id_acidente_dim",
    "id_condicoes"
]

display(
    fato.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in chaves
    ])
)

Registros na fato: 342624
IDs distintos: 342624


id_data,id_localizacao,id_via,id_acidente_dim,id_condicoes
0,0,0,0,0


In [0]:
tabelas_gold = [
    "fato_acidentes",
    "dim_data",
    "dim_localizacao",
    "dim_via",
    "dim_acidente",
    "dim_condicoes",
    "dim_tracado",
    "ponte_acidente_tracado"
]

for tabela in tabelas_gold:
    qtd = spark.table(f"workspace.default.{tabela}").count()
    print(f"{tabela}: {qtd}")

fato_acidentes: 342624
dim_data: 59739
dim_localizacao: 173380
dim_via: 165129
dim_acidente: 2387
dim_condicoes: 10
dim_tracado: 12
ponte_acidente_tracado: 418561


## Validação Final da Camada Gold

A camada Gold foi validada após a criação da tabela fato, dimensões e tabela ponte.

Foram verificadas:

- quantidade de registros da tabela fato;
- unicidade do identificador das ocorrências;
- integridade das chaves dimensionais;
- persistência das tabelas no Unity Catalog;
- disponibilidade das estruturas criadas para consulta.

A tabela `fato_acidentes` manteve a granularidade de uma linha por ocorrência e preservou os 342.624 registros provenientes da camada Silver.